# The first REDUCE number

What does a one percent cut in Region 3 household demand do to annual emissions?

This notebook runs the smallest possible end-to-end experiment through the engine:
load the 2011 EXIOBASE table, aggregate it to the seven game regions, cut every
product in Region 3's household basket by 1%, and read off the change in emissions
and in spend. It is the first empirical result the repo can produce and it is worth
looking at slowly, because every REDUCE tape will be sized off numbers like these.

Two honest caveats before you read any figure:

- **This is the 2011 world, not the 2050 baseline.** The SSP2 walk to 2050 and
  capital endogenisation (`jobs/build_baseline.py`) are not wired in yet. Intensities
  in 2050 will be lower, so a 2050 cut removes fewer tonnes per euro than a 2011 cut.
- **The 50-year figure below uses a flat deployment curve.** A real tape ramps up,
  so its cumulative is smaller than 51 × the annual delta.

**Before you start:** follow [data/README.md](../data/README.md) to download EXIOBASE
and write `config/config.toml`. Parsing the table and inverting the aggregated system
takes about seven minutes on a laptop; the shock itself takes seconds.

## 1. Load the 2011 table and aggregate to game regions

In [ ]:
from pathlib import Path

import pymrio

from redworlds.config import load_config
from redworlds.engine.regions import aggregate_regions

cfg = load_config()
mrio = pymrio.parse_exiobase3(Path(cfg["data"]["exiobase_path"]) / "IOT_2011_pxp")
world = aggregate_regions(mrio)  # 49 EXIOBASE regions → 7 game regions
world.calc_all()  # the Leontief inverse is computed once, here
list(world.get_regions())

## 2. Each region's consumption footprint

`get_region_emissions` reads pymrio's consumption-based account (`D_cba_reg`): each
region is charged for everything its final demand pulls into production anywhere in
the world, plus the fuel its households burn directly. The regional numbers therefore
add up to the world total with no double counting.

In [ ]:
from redworlds.engine.io_tables import GHG_STRESSOR, get_region_emissions
from redworlds.engine.scoring import total_emissions

GT = 1e12  # kg → Gt
for region in world.get_regions():
    print(f"{region:36s} {get_region_emissions(world, region) / GT:6.2f} Gt CO2e/yr")
print(f"{'World':36s} {total_emissions(world) / GT:6.2f} Gt CO2e/yr")

## 3. Cut Region 3 household demand by one percent

`apply_reduce` scales the chosen products in the region's consumption columns and
re-solves output on the cheap path (the Leontief inverse is reused). Nothing is
re-spent: under rule RE2 the money leaves the model, and `gdp_impact` books it.

In [ ]:
from redworlds.actions.reduce import apply_reduce
from redworlds.engine.io_tables import HOUSEHOLDS
from redworlds.engine.scoring import annual_delta, cumulative_delta, gdp_impact

REGION = "Europe and Central Asia"
every_product = list(world.get_sectors())

shocked = apply_reduce(world, REGION, every_product, pct_reduction=0.01, categories=[HOUSEHOLDS])

delta = annual_delta(world, shocked)  # kg CO2e per year, negative = abatement
spend = gdp_impact(world, shocked)  # million EUR (2011), negative = contraction
print(f"annual delta: {delta / 1e9:,.1f} Mt CO2e/yr")
print(f"spend removed: {spend:,.0f} MEUR/yr")
print(f"intensity of the cut: {-delta / (-spend) / 1e6:.2f} kg CO2e per EUR of 2011 spend")

## 4. In the game's currency: cumulative CO₂ over 2050–2100

The game scores one tape as the cumulative delta over the fifty-year window, pegged to
one "brick" of about 1 Gt. With a flat curve (fully deployed from year one) the
arithmetic is just 51 × the annual delta.

In [ ]:
flat = [1.0] * 51
result = cumulative_delta(delta, flat)
cumulative_gt = result["co2_delta_cumulative"] / GT
print(f"cumulative 2050–2100 at flat deployment: {cumulative_gt:,.2f} Gt CO2e")
print(f"one brick (1 Gt) needs a cut of about {1 / -cumulative_gt:.2f}% of Region 3 household demand")
result["jcurve"][:3]

## 5. What to make of it

When this notebook was first run (2026-09-17) it gave, on the 2011 table:

| Quantity | Value |
|---|---|
| Region 3 consumption footprint | 9.2 Gt CO₂e/yr (world 44.5 Gt) |
| Region 3 household spend | 8.3 trillion EUR (2011 basic prices) |
| Annual delta from a 1% household cut | −59 Mt CO₂e/yr |
| Flat 51-year cumulative | −3.0 Gt CO₂e |
| Cut that equals one brick (1 Gt) | about 0.33% of household demand |
| Intensity of the cut | 0.7 kg CO₂e per EUR removed |

The contract doc's back-of-envelope said "about 0.4–0.5% sustained for 50 years" of
non-capital final demand equals one brick. The engine's first answer lands in the same
place, slightly lower, which is what you would expect from 2011 intensities: they are
dirtier than 2050's will be. Note also that the Region 3 delta equals the world delta
to the tonne. That is consumption-based accounting doing its job: only Region 3's demand
changed, so only Region 3's footprint moves, wherever the factories are.

Things to try next, all one-line changes:

- Cut a named basket instead of everything (Basket A in the contract doc: apparel,
  furniture, electronics, vehicles, air transport, hotels and restaurants, recreation).
- Cut a different region and compare the intensity of the cut. Dirtier grids and
  heavier import baskets give more tonnes per euro.
- Ramp the deployment curve instead of using a flat one and watch the cumulative fall.